# ST-GCN — Stream 1 (Geometric)
`train_stgcn_graph.ipynb`  |  `stress_model` environment

Run cells **in order**. **Cell 6** = single-fold plumbing check (run this first). **Cell 7** = full LOSO (only after Cell 6 passes).

In [1]:
# ── Cell 1  Imports ──────────────────────────────────────────────────────────
import os, sys, warnings, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

# data_loader.py lives in the same folder as the notebooks
sys.path.insert(0, r"C:\ML\StressDetection")
from data_loader import load_windows, loso_splits

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
print("data_loader imported OK")

PyTorch : 2.6.0+cu124
CUDA    : True
Device  : cuda
data_loader imported OK


In [2]:
# ── Cell 2  Configuration ─────────────────────────────────────────────────────

# ── Paths ──────────────────────────────────────────────────────────────────────
DL_PATH   = Path(r"C:\ML\StressDetection\DL")
SAVE_PATH = Path(r"C:\ML\StressDetection")
assert DL_PATH.exists(), f"DL_PATH not found: {DL_PATH}"

# ── Adjacency edge list ────────────────────────────────────────────────────────
# Node → MediaPipe landmark mapping (order fixed at extraction time):
#   0:33   1:133  2:145  3:153  4:158  5:159   right eye ring
#   6:474  7:475  8:476  9:477                  iris ring
#   10:70                                        brow
#   11:13  12:14  13:61  14:291                 lip ring
EDGES = [
    # Eye ring — 6-edge outer loop + 2 EAR vertical pairs (top-bottom)
    (0, 4), (4, 5), (5, 1), (1, 3), (3, 2), (2, 0),
    (5, 2), (4, 3),
    # Iris ring
    (6, 7), (7, 8), (8, 9), (9, 6),
    # Iris → eye corners (gaze encodes iris position vs eye boundary)
    (6, 0), (7, 0), (8, 1), (9, 1),
    # Brow → eye top + eye outer (brow-distance pair + cross-region stress signal)
    (10, 5), (10, 0),
    # Lip ring — vertical + horizontal + all 4 corner pairs
    (11, 12), (13, 14),
    (11, 13), (11, 14), (12, 13), (12, 14),
]
NUM_NODES  = 15
NUM_COORDS = 3    # x, y, z
T_FRAMES   = 140

# ── Training hyperparameters ───────────────────────────────────────────────────
EPOCHS      = 30
BATCH_SIZE  = 128
LR          = 1e-3
SEED        = 42
TRAIN_TASKS = ["T1", "T2", "T3"]

# ── Reproducibility ────────────────────────────────────────────────────────────
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

print("Config loaded.")
print(f"DL path : {DL_PATH}")
print(f"Tasks   : {TRAIN_TASKS}")

Config loaded.
DL path : C:\ML\StressDetection\DL
Tasks   : ['T1', 'T2', 'T3']


In [3]:
# ── Cell 3  Adjacency matrix ──────────────────────────────────────────────────

def build_adjacency(edges, num_nodes, self_loops=True):
    # Convert edge list -> degree-normalised adjacency (D^{-1/2} A D^{-1/2}).
    # Self-loops added so each node also aggregates its own features.
    A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    for i, j in edges:
        A[i, j] = 1.0
        A[j, i] = 1.0                              # symmetric
    if self_loops:
        A += np.eye(num_nodes, dtype=np.float32)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(A.sum(axis=1), 1e-8)))
    A_norm = torch.tensor(D_inv_sqrt @ A @ D_inv_sqrt, dtype=torch.float32)
    print(f"Adjacency : {tuple(A_norm.shape)}  |  "
          f"non-zero entries: {(A_norm > 0).sum().item()} / {num_nodes**2}")
    return A_norm

A_norm = build_adjacency(EDGES, NUM_NODES, self_loops=True)

Adjacency : (15, 15)  |  non-zero entries: 63 / 225


In [4]:
# ── Cell 4  Model definition ──────────────────────────────────────────────────

class SpatialGraphConv(nn.Module):
    # For each output node w: out[w] = sum_v A[v,w] * W * x[v]
    # 1x1 conv applies W to all nodes simultaneously, then einsum mixes neighbours.
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.W = nn.Conv2d(in_ch, out_ch, kernel_size=1)

    def forward(self, x, A):
        # x: (B, C_in, T, V)
        x = self.W(x)                               # (B, C_out, T, V)
        x = torch.einsum("bctv,vw->bctw", x, A)    # neighbour aggregation
        return x                                    # (B, C_out, T, V)


class STGCNBlock(nn.Module):
    # One ST-GCN block:
    #   1. Spatial graph conv  — share info across 15 landmark nodes
    #   2. Temporal conv       — capture dynamics over 140 frames
    #   3. Residual connection + learnable per-edge importance weights
    def __init__(self, in_ch, out_ch, num_nodes, t_kernel=9):
        super().__init__()
        self.gcn = SpatialGraphConv(in_ch, out_ch)
        pad = t_kernel // 2
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=(t_kernel, 1), padding=(pad, 0)),
            nn.BatchNorm2d(out_ch),
        )
        self.residual = (
            nn.Conv2d(in_ch, out_ch, kernel_size=1)
            if in_ch != out_ch else nn.Identity()
        )
        self.relu = nn.ReLU(inplace=True)
        # Learnable per-edge importance: init=1.0 means no initial bias.
        # After training, inspect which anatomical connections mattered most.
        self.edge_importance = nn.Parameter(torch.ones(num_nodes, num_nodes))

    def forward(self, x, A):
        # x: (B, C_in, T, V)
        res = self.residual(x)
        x   = self.gcn(x, A * self.edge_importance)
        x   = self.tcn(x)
        return self.relu(x + res)


class STGCN(nn.Module):
    # 3-block ST-GCN for 15-node face landmark graph.
    # Channel progression: 3 -> 32 -> 64 -> 128, then global avg pool -> FC(2).
    def __init__(self, in_ch, num_classes, A):
        super().__init__()
        self.register_buffer("A", A)   # moves to GPU automatically with .to(device)
        self.bn_in  = nn.BatchNorm2d(in_ch)
        n = NUM_NODES
        self.blocks = nn.ModuleList([
            STGCNBlock(in_ch,  32, n),
            STGCNBlock(32,     64, n),
            STGCNBlock(64,    128, n),
        ])
        self.dropout = nn.Dropout(p=0.5)
        self.fc      = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: (B, T, V, C)  e.g. (32, 140, 15, 3)
        x = x.permute(0, 3, 1, 2)      # -> (B, C, T, V)
        x = self.bn_in(x)
        for block in self.blocks:
            x = block(x, self.A)        # (B, C_out, T, V)
        x = x.mean(dim=[2, 3])         # global avg pool -> (B, 128)
        x = self.dropout(x)
        return self.fc(x)               # (B, 2)


# ── Shape sanity check ────────────────────────────────────────────────────────
_m   = STGCN(NUM_COORDS, 2, A_norm)
_x   = torch.zeros(2, T_FRAMES, NUM_NODES, NUM_COORDS)
_out = _m(_x)
print(f"Output shape : {tuple(_out.shape)}   (expected: (2, 2))")
print(f"Parameters   : {sum(p.numel() for p in _m.parameters()):,}")
del _m, _x, _out

Output shape : (2, 2)   (expected: (2, 2))
Parameters   : 216,715


In [5]:
# ── Cell 5  Helper functions ──────────────────────────────────────────────────
# Data loading and LOSO splitting are handled by data_loader.py (Cell 1).
# This cell defines only the training-side helpers.

def normalize_fold(X_train, X_test):
    # Z-score per coordinate (x, y, z). Fit on TRAIN only — never on test.
    # axes (0,1,2) = windows x frames x nodes;  axis 3 = coordinate (kept).
    mean = X_train.mean(axis=(0, 1, 2), keepdims=True)   # (1,1,1,3)
    std  = X_train.std(axis=(0, 1, 2),  keepdims=True) + 1e-8
    return (X_train - mean) / std, (X_test - mean) / std


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n = 0.0, 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
        n += len(y_b)
    return total_loss / n


def evaluate(model, loader, device):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            preds.extend(model(X_b.to(device)).argmax(1).cpu().numpy())
            labels.extend(y_b.numpy())
    p, l = np.array(preds), np.array(labels)
    return {
        "acc"     : accuracy_score(l, p),
        "bal_acc" : balanced_accuracy_score(l, p),
        "f1"      : f1_score(l, p, zero_division=0),
    }


def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)


def class_weights(y_train):
    # Inverse-frequency weights. With 2:1 imbalance, calm (label 0) gets 2x weight.
    counts = np.bincount(y_train)
    return torch.tensor(1.0 / counts, dtype=torch.float32)


print("Helper functions defined.")

Helper functions defined.


In [6]:
# ── Cell 6  Single-fold plumbing check (test pid = 15) ────────────────────────
# Run this first. If loss decreases and bal_acc is above 0.5, proceed to Cell 7.

TEST_PID = 15

# load_windows discovers p{N} subfolders under DL_PATH automatically
X, y, pids = load_windows(DL_PATH, file_type="graph", train_tasks=TRAIN_TASKS)

# Manual split for the plumbing check — no loop overhead
test_mask  = (pids == TEST_PID)
train_mask = ~test_mask

X_test_r, y_test   = X[test_mask],  y[test_mask]
X_train_r, y_train = X[train_mask], y[train_mask]

# Normalize: fit on train, apply to both
X_train, X_test = normalize_fold(X_train_r, X_test_r)

print(f"Train : {len(X_train)} windows  |  class dist: {np.bincount(y_train)}")
print(f"Test  : {len(X_test)}  windows  |  class dist: {np.bincount(y_test)}")

loader_tr = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
loader_te = make_loader(X_test,  y_test,  BATCH_SIZE, shuffle=False)

model     = STGCN(NUM_COORDS, 2, A_norm).to(device)
weights   = class_weights(y_train).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print(f"Parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Device     : {device}")
print()

for ep in range(1, EPOCHS + 1):
    loss = train_epoch(model, loader_tr, optimizer, criterion, device)
    scheduler.step()
    if ep % 5 == 0 or ep == 1:
        m = evaluate(model, loader_te, device)
        print(f"Epoch {ep:3d}  loss={loss:.4f}  "
              f"acc={m['acc']:.3f}  bal_acc={m['bal_acc']:.3f}  f1={m['f1']:.3f}")

print()
print("Single-fold check done.")
print("Loss should decrease. bal_acc should be above 0.5. If both hold -> run Cell 7.")

LOADING file_type='graph'  tasks=['T1', 'T2', 'T3']
Source: C:\ML\StressDetection\DL
Found 56 participant folders
----------------------------------------------------------------
----------------------------------------------------------------
Files loaded   : 168
Files missing  : 0
Total windows  : 29735
Window shape   : (140, 15, 3)
Label 0 (calm) : 9911
Label 1 (stress): 19824
Class balance  : 2.00 : 1  (stress : calm)
Participants   : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56]


Train : 29204 windows  |  class dist: [ 9734 19470]
Test  : 531  windows  |  class dist: [177 354]
Parameters : 216,715
Device     : cuda

Epoch   1  loss=0.5497  acc=0.864  bal_acc=0.898  f1=0.887
Epoch   5  loss=0.2905  acc=0.650  bal_acc=0.487  f1=0.788
Epoch  10  loss=0.1997  acc=0.932  bal_acc=0.922  f1=0.949
Epoch  15  loss=0.1366  acc=0.964  bal_acc=0.973  f1=0.972
Epoch  20  loss=0.1175  acc=0.846  bal_acc=0.832  f1=0.883
Epoch  25  loss=0.0751  acc=0.927  bal_acc=0.945  f1=0.942
Epoch  30  loss=0.0683  acc=0.851  bal_acc=0.850  f1=0.884

Single-fold check done.
Loss should decrease. bal_acc should be above 0.5. If both hold -> run Cell 7.


In [6]:
# ── Cell 7  Full LOSO ─────────────────────────────────────────────────────────
# Run ONLY after Cell 6 passes. Each fold trains a fresh model from scratch.
# loso_splits handles fold generation, empty-fold skipping, and per-fold logging.

X, y, pids = load_windows(DL_PATH, file_type="graph", train_tasks=TRAIN_TASKS)
results    = []
n_total    = len(set(pids.tolist()))   # computed once, used in fold counter

print(f"Starting LOSO  |  {n_total} participants  |  {EPOCHS} epochs each")
print()

for fold_i, (X_train_r, y_train, X_test_r, y_test, test_pid) in \
        enumerate(loso_splits(X, y, pids, verbose=False)):

    X_train, X_test = normalize_fold(X_train_r, X_test_r)

    loader_tr = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
    loader_te = make_loader(X_test,  y_test,  BATCH_SIZE, shuffle=False)

    model     = STGCN(NUM_COORDS, 2, A_norm).to(device)
    weights   = class_weights(y_train).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    for ep in range(1, EPOCHS + 1):
        train_epoch(model, loader_tr, optimizer, criterion, device)
        scheduler.step()

    m = evaluate(model, loader_te, device)
    results.append({"pid": test_pid, **m})
    print(f"[{fold_i+1:2d}/{n_total}] pid={test_pid:3d}  "
          f"acc={m['acc']:.3f}  bal_acc={m['bal_acc']:.3f}  f1={m['f1']:.3f}")

# ── Summary ───────────────────────────────────────────────────────────────────
df = pd.DataFrame(results)
print()
print("=" * 62)
print(f"ST-GCN LOSO   |   {len(df)} folds   |   {n_total} participants")
print("=" * 62)
for col, label in [("acc","Accuracy"),("bal_acc","Balanced Acc"),("f1","F1 (stress)")]:
    print(f"  {label:<16}: {df[col].mean():.3f} +/- {df[col].std():.3f}")
print("=" * 62)
print()
print("Honest baseline to beat : 64.6%  (Koceska & Koceski 2021)")
print("ML reference (XGBoost)  : ~92.0% (geometric stream, full 56 participants)")

out_path = SAVE_PATH / "stgcn_loso_results.csv"
df.to_csv(out_path, index=False)
print(f"Results saved -> {out_path}")

LOADING file_type='graph'  tasks=['T1', 'T2', 'T3']
Source: C:\ML\StressDetection\DL
Found 56 participant folders
----------------------------------------------------------------
----------------------------------------------------------------
Files loaded   : 168
Files missing  : 0
Total windows  : 29735
Window shape   : (140, 15, 3)
Label 0 (calm) : 9911
Label 1 (stress): 19824
Class balance  : 2.00 : 1  (stress : calm)
Participants   : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56]


Starting LOSO  |  56 participants  |  30 epochs each

[ 1/56] pid=  1  acc=0.714  bal_acc=0.571  f1=0.823
[ 2/56] pid=  2  acc=0.983  bal_acc=0.987  f1=0.987
[ 3/56] pid=  3  acc=0.874  bal_acc=0.883  f1=0.900
[ 4/56] pid=  4  acc=0.795  bal_acc=0.694  f1=0.866
[ 5/56] pid=  5  acc=0.702  bal_acc=0.572  f1=0.812
[ 6/56] pid=  6  acc=0.944  bal_acc=0.958  f1=0.956
[ 7/56] pid=  7  acc=0.667  bal_acc=0.500  f1=0.800
[ 8/56] pid=  8  acc=0.970  bal_acc=0.955  f1=0.978
[ 9/56] pid=  9  acc=0.766  bal_acc=0.754  f1=0.819
[10/56] pid= 10  acc=0.729  bal_acc=0.593  f1=0.831
[11/56] pid= 11  acc=0.710  bal_acc=0.565  f1=0.821
[12/56] pid= 12  acc=0.812  bal_acc=0.718  f1=0.876
[13/56] pid= 13  acc=0.895  bal_acc=0.842  f1=0.927
[14/56] pid= 14  acc=0.944  bal_acc=0.942  f1=0.957
[15/56] pid= 15  acc=0.782  bal_acc=0.754  f1=0.836
[16/56] pid= 16  acc=0.849  bal_acc=0.774  f1=0.898
[17/56] pid= 17  acc=0.667  bal_acc=0.500  f1=0.800
[18/56] pid= 18  acc=0.968  bal_acc=0.969  f1=0.976
[19/56] pi

In [9]:
torch.cuda.mem_get_info()

(3077570560, 6441926656)